In [ ]:
!python -c "import sys; print(sys.executable)"
import sys
# !{sys.executable} -m pip install numpy

import numpy, torch
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

#1. load the document
loader = PyPDFLoader(file_path="attention.pdf")
pdfDoc = loader.load()

#2. Chunks
seperators = ["\n\n","\n"," ",""]
splitter = RecursiveCharacterTextSplitter(separators=seperators,chunk_size = 1000,chunk_overlap=50)
pdfChunks = splitter.split_documents(pdfDoc)
print(len(pdfChunks))

#3. embedding
embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#4. vector store
vectorStore = FAISS.from_documents(documents=pdfChunks,embedding=embed)
# answer = vectorStore.similarity_search(query="What is attention mechanism?",k=1)
# print(answer)

#5. save
vectorStore.save_local(folder_path="vector_store",index_name="ex")

#6. load vector store
faissVectorStore = FAISS.load_local(folder_path="vector_store",embeddings=embed,allow_dangerous_deserialization=True,index_name="ex")















48


/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain_langraph/langchain_langgraph/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# 7. prepare chat tempalte
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(  [
            ("system", "Answer the question based only on the provided context.\n\nContext:\n{context}"),
            ("human", "{input}"),
        ])

# template.invoke({"question":"What is attention"})



In [ ]:
# 8. get llm
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(repo_id="meta-llama/Llama-3.2-3B-Instruct",temperature=0)
chat_model = ChatHuggingFace(llm = llm)


In [ ]:
# 9. get the retriever
base_retriever = faissVectorStore.as_retriever(search_type ="similarity",search_kwargs = {"k":3})
test_docs = base_retriever.invoke("what is attention?")
print(f"Retrieved {len(test_docs)} documents.")




In [ ]:
#10 prepare chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser

doc_chain = create_stuff_documents_chain(llm = chat_model,prompt=template,output_parser=StrOutputParser())

ret_chain = create_retrieval_chain(retriever=base_retriever, combine_docs_chain=doc_chain)

ret_chain.invoke(input = {"input":"what is attention?"})

In [ ]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient( api_key=os.environ["HF_TOKEN"])
resp = client.chat_completion(
    model="meta-llama/Llama-3.2-3B-Instruct",
    messages=[{"role": "user", "content": "hi"}],
    max_tokens=20,
)
print(resp.choices[0].message.content)


In [ ]:
print("Assistant: Hello! Ask me anything about the attention mechanism paper.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ("exit", "quit"):
        break
    result = ret_chain.invoke({"input": user_input})
    print("Assistant:", result["answer"], "\n")